In [1]:
import os
import logging

# 1. Force C++ backend to only show FATAL errors
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"

# 2. Silence Python's absl logging module before importing TensorFlow
import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)

# 3. Silence Python's standard logging for TensorFlow
logging.getLogger('tensorflow').setLevel(logging.FATAL)

import tensorflow as tf

In [2]:
import sys

REPO_NAME = "RefraScan"
GITHUB_USER = "KyziaPi"
BRANCH_NAME = "ResNet50-Train"   # <-- point this at your branch

REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
REPO_PATH = os.path.join("/kaggle/working", REPO_NAME)

if not os.path.exists(REPO_NAME):
    print(f"Cloning branch '{BRANCH_NAME}' from {REPO_NAME}...")
    !env GIT_TERMINAL_PROMPT=0 git clone -b {BRANCH_NAME} {REPO_URL}
else:
    print(f"{REPO_NAME} already exists. Switching branch and pulling latest updates...")
    !cd {REPO_NAME} && env GIT_TERMINAL_PROMPT=0 git fetch --all && git checkout {BRANCH_NAME} && git pull origin {BRANCH_NAME}

if os.path.exists(REPO_PATH):
    if REPO_PATH not in sys.path:
        sys.path.append(REPO_PATH)

    import math
    import pandas as pd
    from tensorflow.keras.applications.resnet50 import preprocess_input

    try:
        from src.preprocessing import load_and_clean_data
        from src.cross_validation import run_cross_validation
        print("🚀 Success! Custom modules imported smoothly.")
    except ModuleNotFoundError as e:
        print(f"❌ Still failing. Current sys.path contains: {sys.path}")
        raise e

    print(f"Environment configured successfully! Working on branch: {BRANCH_NAME}")
else:
    print("❌ Error: Repository failed to clone.")

Cloning branch 'ResNet50-Train' from RefraScan...
Cloning into 'RefraScan'...
remote: Enumerating objects: 294, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 294 (delta 0), reused 1 (delta 0), pack-reused 288 (from 2)
Receiving objects: 100% (294/294), 43.93 MiB | 36.93 MiB/s, done.
Resolving deltas: 100% (129/129), done.
🚀 Success! Custom modules imported smoothly.
Environment configured successfully! Working on branch: ResNet50-Train


In [4]:
#QUICK SANITY CHECK using minimal folds + minimal epochs to catch errors before the full run.
DATASET_DIR = '/kaggle/input/datasets/yerikaelainegueco/fundus-images-with-refractive-values' 
CSV_PATH = os.path.join(DATASET_DIR, 'RefraScan_dataset.csv')
IMG_DIR = os.path.join(DATASET_DIR, 'FundusImages')

df = load_and_clean_data(CSV_PATH, IMG_DIR)

df['classification_encoded'] = df['classification'].map(
    {'Emmetropia': 0, 'Myopia': 1, 'Hyperopia': 2}
)

test_results = run_cross_validation(
    df=df,
    model_name='resnet50',
    preprocess_input=preprocess_input,
    patient_col="ID",
    target_col="classification_encoded",
    n_splits=2,              # minimum allowed by StratifiedGroupKFold
    batch_size=16,
    epochs=2,                # confirm training runs end-to-end
    learning_rate=0.0001,
    holdout_test_size=0.15,
    fine_tune=True,          # <-- important: tests the NEW code path too
    fine_tune_epochs=2,
    fine_tune_lr=1e-5,
)

Dataset Split: 866 samples for 10-Fold CV | 152 samples in Holdout Test Set (15%)

STARTING 2-FOLD CROSS VALIDATION


--- Fold 1/2 ---


I0000 00:00:1785667696.486213      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1785667696.489130      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/2
 1/52 ━━━━━━━━━━━━━━━━━━━━ 15:01 18s/step - accuracy: 0.3750 - loss: 1.6117

I0000 00:00:1785667719.656376     146 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 531ms/step - accuracy: 0.4173 - loss: 0.9311
Epoch 1: val_loss improved from None to 0.38092, saving model to best_resnet50_fold_1.h5

Epoch 1: finished saving model to best_resnet50_fold_1.h5
52/52 ━━━━━━━━━━━━━━━━━━━━ 68s 977ms/step - accuracy: 0.4663 - loss: 0.7208 - val_accuracy: 0.6874 - val_loss: 0.3809
Epoch 2/2
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 475ms/step - accuracy: 0.5264 - loss: 0.5387
Epoch 2: val_loss improved from 0.38092 to 0.37512, saving model to best_resnet50_fold_1.h5

Epoch 2: finished saving model to best_resnet50_fold_1.h5
52/52 ━━━━━━━━━━━━━━━━━━━━ 34s 661ms/step - accuracy: 0.5385 - loss: 0.5156 - val_accuracy: 0.6391 - val_loss: 0.3751
Restoring model weights from the end of the best epoch: 2.
Unfroze 7 layers: ['conv5_block3_1_conv', 'conv5_block3_1_relu', 'conv5_block3_2_conv', 'conv5_block3_2_relu', 'conv5_block3_3_conv', 'conv5_block3_add', 'conv5_block3_out']
Epoch 1/2
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 478ms/step - accuracy: 0.5424 - 

In [5]:
DATASET_DIR = '/kaggle/input/datasets/yerikaelainegueco/fundus-images-with-refractive-values' 
CSV_PATH = os.path.join(DATASET_DIR, 'RefraScan_dataset.csv')
IMG_DIR = os.path.join(DATASET_DIR, 'FundusImages')

df = load_and_clean_data(CSV_PATH, IMG_DIR)

df['classification_encoded'] = df['classification'].map(
    {'Emmetropia': 0, 'Myopia': 1, 'Hyperopia': 2}
)

results = run_cross_validation(
    df=df,
    model_name='resnet50',
    preprocess_input=preprocess_input,
    patient_col="ID",
    target_col="classification_encoded",
    n_splits=10,
    batch_size=16,
    epochs=30,
    learning_rate=0.0001,
    holdout_test_size=0.15,
    fine_tune=True,
    fine_tune_epochs=15,
    fine_tune_lr= 1e-5,
)

Dataset Split: 866 samples for 10-Fold CV | 152 samples in Holdout Test Set (15%)

STARTING 10-FOLD CROSS VALIDATION


--- Fold 1/10 ---
Epoch 1/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 485ms/step - accuracy: 0.4308 - loss: 0.9642
Epoch 1: val_loss improved from None to 0.30495, saving model to best_resnet50_fold_1.h5

Epoch 1: finished saving model to best_resnet50_fold_1.h5
96/96 ━━━━━━━━━━━━━━━━━━━━ 70s 602ms/step - accuracy: 0.4850 - loss: 0.7402 - val_accuracy: 0.7558 - val_loss: 0.3050
Epoch 2/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 468ms/step - accuracy: 0.5171 - loss: 0.5602
Epoch 2: val_loss improved from 0.30495 to 0.30379, saving model to best_resnet50_fold_1.h5

Epoch 2: finished saving model to best_resnet50_fold_1.h5
96/96 ━━━━━━━━━━━━━━━━━━━━ 46s 487ms/step - accuracy: 0.5449 - loss: 0.5254 - val_accuracy: 0.7442 - val_loss: 0.3038
Epoch 3/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 437ms/step - accuracy: 0.5525 - loss: 0.5012
Epoch 3: val_loss improved from 0.30379 to 0.30271, saving model to be